# Asynchronous simulation for BMA

A short notebook exploring the dynamics of asynchronous networks

## load the libraries

In [1]:
import pybma

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np

from copy import deepcopy

## Define some functions for simulation and analysis

In [2]:
class StateEncoder:
    def __init__(self, model):
        maxes     = [v['RangeTo'] for v in model['Model']['Variables']]
        variables = [v['Id']      for v in model['Model']['Variables']]

        sorted_pairs    = sorted(zip(variables, maxes), key=lambda x: x[0])
        self.variables  = [v for v, _ in sorted_pairs]
        self.max_levels = [m for _, m in sorted_pairs]

        # Pure Python ints — no numpy, no overflow
        self.strides = [1]
        for m in self.max_levels[:-1]:
            self.strides.append(self.strides[-1] * (m + 1))

    def encode(self, state: tuple) -> int:
        return sum(v * s for v, s in zip(state, self.strides))

    def decode(self, code: int) -> tuple:
        result = []
        for m in self.max_levels:
            result.append(code % (m + 1))
            code //= (m + 1)
        return tuple(result)

In [3]:
class BNContext:
    """
    Holds the canonical variable ordering derived from the model.
    All state conversions go through this object, eliminating
    ad-hoc variable list construction elsewhere.
    """
    def __init__(self, model):
        self.encoder  = StateEncoder(model)
        self.variables  = self.encoder.variables    # same sorted list, single source
        self.max_levels = self.encoder.max_levels   # same sorted list, single source

    def stateToTuple(self, state: dict) -> tuple:
        return tuple(state[v] for v in self.variables)

    def tupleToState(self, t: tuple) -> dict:
        return dict(zip(self.variables, t))

    def async_step(self, qn, state: dict) -> set[tuple]:
        sync_result = pybma.simulate(qn, steps=3, initial_values=state)
        sync_target = {v: sync_result[v][-1] for v in self.variables}
        
        async_result = set()
        for v in self.variables:
            if sync_target[v] != state[v]:
                successor = dict(state)
                successor[v] = sync_target[v]
                async_result.add(self.stateToTuple(successor))
        return async_result

In [4]:
from collections import deque

def exists_order_before(
    graph: dict[int, set[int]],
    ctx: BNContext,
    start: tuple,
    i: int, v: int,
    j: int, w: int
) -> bool:
    """
    Returns True if there exists a path from start where
    gene i takes value v at some state BEFORE gene j takes value w.
    """
    start_code = ctx.encoder.encode(start)
    
    visited = set()
    queue = deque()
    queue.append((start_code, ctx.encoder.decode(start_code)[i] == v))
    
    while queue:
        code, i_seen = queue.popleft()
        
        if (code, i_seen) in visited:
            continue
        visited.add((code, i_seen))
        
        state = ctx.encoder.decode(code)
        
        if state[j] == w and not i_seen:
            continue
        
        if i_seen and state[j] != w:
            return True
        
        for successor_code in graph.get(code, set()):
            new_i_seen = i_seen or (ctx.encoder.decode(successor_code)[i] == v)
            queue.append((successor_code, new_i_seen))
    
    return False

In [5]:
def exists_order_before_online(
    qn: BNModel,
    ctx: BNContext,
    start: tuple,
    i: int, v: int,
    j: int, w: int
) -> tuple[bool, list[tuple] | None]:
    """
    Returns (True, trace) if there exists a path from start where
    gene i takes value v before gene j takes value w, or (False, None).
    trace is a list of decoded state tuples from start to the witnessing state.
    """
    start_code = ctx.encoder.encode(start)
    initial_i_seen = start[i] == v
        
    visited = set()
    # parent maps (code, i_seen) -> (parent_code, parent_i_seen) | None
    parent: dict[tuple[int, bool], tuple[int, bool] | None] = {}

    queue = deque()
    queue.append((start_code, initial_i_seen))
    parent[(start_code, initial_i_seen)] = None

    while queue:
        code, i_seen = queue.popleft()

        if (code, i_seen) in visited:
            continue
        visited.add((code, i_seen))

        state = ctx.encoder.decode(code)

        if state[j] == w and not i_seen:
            continue

        if i_seen and state[j] != w:
            return True, _reconstruct_trace(parent, ctx, code, i_seen)

        #print("P:",state[i],state[j])
        for successor in ctx.async_step(qn, ctx.tupleToState(state)):
            #print("S:",successor[i],successor[j])
            successor_code = ctx.encoder.encode(successor)
            new_i_seen = i_seen or (successor[i] == v)
            key = (successor_code, new_i_seen)
            if key not in parent:
                parent[key] = (code, i_seen)
            queue.append((successor_code, new_i_seen))

    return False, None


def _reconstruct_trace(
    parent: dict[tuple[int, bool], tuple[int, bool] | None],
    ctx: BNContext,
    code: int,
    i_seen: bool
) -> list[tuple]:
    """Walk parent pointers back to the start and reverse."""
    trace = []
    cursor: tuple[int, bool] | None = (code, i_seen)
    while cursor is not None:
        trace.append(ctx.encoder.decode(cursor[0]))
        cursor = parent[cursor]
    trace.reverse()
    return trace

In [6]:
def async_simulate(qn,initial_values=None, ctx=None):
    if initial_values == None or ctx == None: raise
    
    variables = list(initial_values.keys())
    initial_state = ctx.stateToTuple(initial_values)
    
    graph: dict[int, set[int]] = {}

    # Adding a transition
    def add_transition(graph, from_state, to_state):
        graph.setdefault(ctx.encoder.encode(from_state), set()).add(ctx.encoder.encode(to_state))
    
    final_values = [initial_state]
    stop = False
    while not stop:
        stop = True
        updated_final_values = []
        for final_values_state in final_values:
            if ctx.encoder.encode(final_values_state) in graph.keys():
                continue
            stop = False
            final_values_single = ctx.tupleToState(final_values_state)
            step_result = ctx.async_step(qn,final_values_single)
            updated_final_values.append(step_result)
            for successor in step_result:
                add_transition(graph,final_values_state,successor)
        final_values = [x for sublist in updated_final_values for x in sublist]  
    return(graph)

In [7]:
def report_transitions(trace: list[tuple]) -> None:
    """
    Given a list of state tuples (e.g. from exists_order_before_online),
    prints the indices that change and by how much at each step.
    """
    for step, (src, tgt) in enumerate(zip(trace, trace[1:])):
        diffs = [(i, src[i], tgt[i], tgt[i] - src[i])
                 for i in range(len(src))
                 if src[i] != tgt[i]]
        print(f"Step {step}: {src} -> {tgt}")
        for i, old, new, delta in diffs:
            print(f"  index {i}: {old} -> {new}  (delta {delta:+d})")
        if len(diffs) != 1:
            print(f"  *** WARNING: {len(diffs)} genes changed (expected 1 for async) ***")

In [40]:
def report_trace(trace: list[tuple], ctx: BNContext, model: dict) -> None:
    """
    Reports a trace as a sequence of named gene changes.
    """
    # Build ID -> name lookup from model
    id_to_name = {v['Id']: v['Name'] for v in model['Model']['Variables']}
    # Map ctx.variables (sorted IDs) to names in the same order
    names = [id_to_name[v] for v in ctx.variables]

    print(f"Trace length: {len(trace)} states, {len(trace)-1} transitions")
    print(f"Start: { {names[i]: trace[0][i] for i in range(len(names))} }")
    print()

    for step, (src, tgt) in enumerate(zip(trace, trace[1:])):
        diffs = [(i, src[i], tgt[i]) for i in range(len(src)) if src[i] != tgt[i]]

        if len(diffs) == 1:
            i, old, new = diffs[0]
            print(f"Step {step+1}: {names[i]}  {old} -> {new}")
        elif len(diffs) == 0:
            print(f"Step {step+1}: *** WARNING: no change (self-loop) ***")
        else:
            changed = ", ".join(f"{names[i]}: {old}->{new}" for i, old, new in diffs)
            print(f"Step {step+1}: *** WARNING: {len(diffs)} genes changed: {changed} ***")

    print()
    print(f"End:   { {names[i]: trace[-1][i] for i in range(len(names))} }")

## load the model and check the stability

In [8]:
m = pybma.load_model("./ToyModelStable.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)
print(p)

{'ProofProgression': {'stable': True, 'history': [(4, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (3, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (2, {1: (0, 0), 2: (0, 0), 3: (0, 0)}), (1, {1: (0, 0), 2: (0, 0), 3: (0, 4)}), (0, {1: (0, 0), 2: (0, 4), 3: (0, 4)})]}, 'CounterExample': None}


In [9]:
final_state = dict( [ (varid,p['ProofProgression']['history'][0][1][varid][0]) for varid in p['ProofProgression']['history'][0][1].keys() ] )

In [10]:
# Some issue with number of steps; 3 gives us two states
pybma.simulate(qn,steps=3,initial_values=final_state)

{1: [0, 0], 2: [0, 0], 3: [0, 0]}

## Mutate the model

In [11]:
m['Model']['Variables'][0]['Formula'] = "2"
qM = pybma.model_to_qn(m)
pM = pybma.check_stability(qM)

print("###Modified Model###")
print(pM)

###Modified Model###
{'ProofProgression': {'stable': True, 'history': [(4, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (3, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (2, {1: (2, 2), 2: (2, 2), 3: (2, 2)}), (1, {1: (2, 2), 2: (2, 2), 3: (0, 4)}), (0, {1: (2, 2), 2: (0, 4), 3: (0, 4)})]}, 'CounterExample': None}


### Simulate the new model from the old stable state

In [12]:
pybma.simulate(qM,steps=3,initial_values=final_state)

{1: [0, 1], 2: [0, 0], 3: [0, 0]}

In [13]:
ctx = BNContext(m)

In [14]:
ctx.async_step(qM,final_state)

{(1, 0, 0)}

In [15]:
graph = async_simulate(qM,final_state,ctx)
print(graph)

{0: {1}, 1: {2, 6}, 6: {31, 7}, 2: {7}, 7: {32, 12}, 31: {32}, 32: {37}, 12: {37}, 37: {62}}


In [16]:
variables = list(final_state.keys())



In [17]:
exists_order_before(graph, ctx, (0, 0, 0), 0, 1, 1, 1)

True

In [18]:
exists_order_before(graph, ctx, (0, 0, 0), 1, 1, 0, 1)

False

In [19]:
variables = list(final_state.keys())
exists_order_before_online(qM, ctx, (0, 0, 0), 0, 2, 1, 1)

(True, [(0, 0, 0), (1, 0, 0), (2, 0, 0)])

In [20]:
exists_order_before_online(qM, ctx, (0, 0, 0), 1, 1, 0, 1)

(False, None)

In [21]:
finding, trace = exists_order_before_online(qM, ctx, (0, 0, 0), 0, 2, 1, 1)
report_transitions(trace)

Step 0: (0, 0, 0) -> (1, 0, 0)
  index 0: 0 -> 1  (delta +1)
Step 1: (1, 0, 0) -> (2, 0, 0)
  index 0: 1 -> 2  (delta +1)


## B cell differentiation

In [22]:
m = pybma.load_model("./B-cell-undifferentiated-stable.json")
qn = pybma.model_to_qn(m)
p = pybma.check_stability(qn)
print("Stable =", p['ProofProgression']['stable'])

Stable = True


In [23]:
final_state = dict( [ (varid,p['ProofProgression']['history'][0][1][varid][0]) for varid in p['ProofProgression']['history'][0][1].keys() ] )

In [24]:
variable_names_lookup = { v['Id']:v['Name'] for v in m['Model']['Variables'] }
variable_ids_lookup = { v['Name']:v['Id'] for v in m['Model']['Variables'] }
print('CLP',variable_ids_lookup['CLP'])
print('Naive B cell',variable_ids_lookup['Naive B cell'])

CLP 146
Naive B cell 164


In [25]:
print(final_state)

{2: 0, 3: 0, 7: 0, 8: 0, 11: 0, 17: 0, 19: 0, 30: 0, 38: 0, 59: 2, 68: 0, 74: 0, 120: 1, 121: 1, 122: 1, 126: 0, 130: 0, 133: 0, 146: 1, 147: 0, 148: 0, 149: 0, 164: 0, 171: 0, 179: 0, 180: 0, 181: 0, 182: 0, 198: 0, 206: 0, 211: 0, 231: 2, 232: 0, 236: 0, 237: 0, 247: 0, 254: 0, 257: 0, 266: 0, 272: 1, 275: 0, 280: 1, 293: 0, 307: 0, 313: 0, 322: 0, 331: 0, 343: 0, 344: 0, 366: 0, 383: 0, 390: 0, 392: 0, 408: 0, 414: 0, 415: 0, 423: 0, 424: 0, 425: 0, 437: 0, 457: 0, 464: 0, 471: 0, 472: 0, 473: 0}


In [26]:
print("CLP final =", final_state[variable_ids_lookup['CLP']])
print("Naive B cell final =", final_state[variable_ids_lookup['Naive B cell']])

CLP final = 1
Naive B cell final = 0


In [27]:
m['Model']['Variables'][0]

{'Name': 'Ilr7',
 'Id': 2,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'min(2,(2*var(7)))+var(74)+var(8)-(4*var(30))-max(0,(2*var(206)-var(211)))'}

In [28]:

varlist = [v['Name'] for v in m['Model']['Variables'] ]
variableDict = dict(zip(varlist,range(len(varlist))))

In [29]:
variableDict

{'Ilr7': 0,
 'Il7': 1,
 'E2a': 2,
 'Foxo1': 3,
 'Pax5': 4,
 'Ikzf1': 5,
 'CD19': 6,
 'BCR': 7,
 'RAG': 8,
 'RUNX1': 9,
 'Vpreb1': 10,
 'EBF1': 11,
 'Proliferation': 12,
 'Apoptosis': 13,
 'Cycle-arrest': 14,
 'CCND3': 15,
 'Bcl2': 16,
 'TSLP': 17,
 'CLP': 18,
 'PreproB': 19,
 'ProB': 20,
 'PreB': 21,
 'Naive B cell': 22,
 'JAK-STAT': 23,
 'Syk': 24,
 'Bcl6': 25,
 'CDKN1A': 26,
 'CDKN1B': 27,
 'Bok': 28,
 'preBCR': 29,
 'Bach2': 30,
 'CXCL12': 31,
 'CXCR4': 32,
 'Id3': 33,
 'Ikzf3': 34,
 'Igll1': 35,
 'CRLF2': 36,
 'Myc': 37,
 'PIP3': 38,
 'CCND2': 39,
 'BCR-ABL': 40,
 'CDKN2A': 41,
 'preBCR_Pax5': 42,
 'AKT': 43,
 'DNA-damage': 44,
 'Bim': 45,
 'p53': 46,
 'MAPK': 47,
 'ERK1_2': 48,
 'Blnk': 49,
 'CXCR4_CXCL12': 50,
 'KO_preBCR': 51,
 'KO_JAK_STAT': 52,
 'IKZF1_ko': 53,
 'BCL6_KO': 54,
 'MYC_KO': 55,
 'KO_CDKN1A': 56,
 'KO_CDKN1B': 57,
 'KO_CDKN2A': 58,
 'Force-Pax5': 59,
 'EBF1_KO': 60,
 'PAX5_KO': 61,
 'KO_PIP3': 62,
 'KO_MAPK': 63,
 'KO_BCRABL': 64}

In [30]:
m['Model']['Variables'][variableDict['PreproB']]

{'Name': 'PreproB',
 'Id': 147,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'min(1,var(2))-max(0,(2*var(74)-2))-max(0,2*var(206)-2)-var(30)'}

In [31]:
m['Model']['Variables'][variableDict['CLP']]

{'Name': 'CLP',
 'Id': 146,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': '1-var(2)-var(30)-var(206)'}

In [42]:
m['Model']['Variables'][variableDict['ProB']]

{'Name': 'ProB',
 'Id': 148,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'min(1,2*var(74)-2)-max(0,2*var(206)-2)-var(30)'}

In [43]:
m['Model']['Variables'][variableDict['PreB']]

{'Name': 'PreB',
 'Id': 149,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'max(0,2*var(206)-3)-max(0,2*var(30)-2)'}

In [44]:
m['Model']['Variables'][variableDict['Naive B cell']]

{'Name': 'Naive B cell',
 'Id': 164,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': '2*var(30)-3'}

In [32]:
m['Model']['Variables'][variableDict['Il7']]['RangeTo'] = 2
m['Model']['Variables'][variableDict['Ikzf1']]['RangeTo'] = 2
qM = pybma.model_to_qn(m)


In [45]:
ctx = BNContext(m)
final_state_tuple = ctx.stateToTuple(final_state)
clpVar = ctx.variables.index(146)
preprobVar = ctx.variables.index(147)
probVar = ctx.variables.index(148)
prebVar = ctx.variables.index(149)
naivebVar = ctx.variables.index(164)


In [46]:
finding, trace = exists_order_before_online(qM, ctx, final_state_tuple, probVar, 1, preprobVar, 1)
print(finding, trace)

True [(0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 2, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (1, 2, 1,

In [47]:
report_trace(trace, ctx, m)

Trace length: 13 states, 12 transitions
Start: {'Ilr7': 0, 'Il7': 0, 'E2a': 0, 'Foxo1': 0, 'Pax5': 0, 'Ikzf1': 0, 'CD19': 0, 'BCR': 0, 'RAG': 0, 'RUNX1': 2, 'Vpreb1': 0, 'EBF1': 0, 'Proliferation': 1, 'Apoptosis': 1, 'Cycle-arrest': 1, 'CCND3': 0, 'Bcl2': 0, 'TSLP': 0, 'CLP': 1, 'PreproB': 0, 'ProB': 0, 'PreB': 0, 'Naive B cell': 0, 'JAK-STAT': 0, 'Syk': 0, 'Bcl6': 0, 'CDKN1A': 0, 'CDKN1B': 0, 'Bok': 0, 'preBCR': 0, 'Bach2': 0, 'CXCL12': 2, 'CXCR4': 0, 'Id3': 0, 'Ikzf3': 0, 'Igll1': 0, 'CRLF2': 0, 'Myc': 0, 'PIP3': 0, 'CCND2': 1, 'BCR-ABL': 0, 'CDKN2A': 1, 'preBCR_Pax5': 0, 'AKT': 0, 'DNA-damage': 0, 'Bim': 0, 'p53': 0, 'MAPK': 0, 'ERK1_2': 0, 'Blnk': 0, 'CXCR4_CXCL12': 0, 'KO_preBCR': 0, 'KO_JAK_STAT': 0, 'IKZF1_ko': 0, 'BCL6_KO': 0, 'MYC_KO': 0, 'KO_CDKN1A': 0, 'KO_CDKN1B': 0, 'KO_CDKN2A': 0, 'Force-Pax5': 0, 'EBF1_KO': 0, 'PAX5_KO': 0, 'KO_PIP3': 0, 'KO_MAPK': 0, 'KO_BCRABL': 0}

Step 1: Il7  0 -> 1
Step 2: Il7  1 -> 2
Step 3: Ikzf1  0 -> 1
Step 4: E2a  0 -> 1
Step 5: Ilr7  0 -> 1
S

In [57]:
m['Model']['Variables'][variableDict['EBF1']]

{'Name': 'EBF1',
 'Id': 74,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': '((floor(((var(8)+var(59)+var(11)+var(171)+var(457))/5)+1/3)+(2*var(7)-2)-max(0,(2*var(206)\n-var(211)))-var(237)/2)*((2-var(457))/2))\n+ (min(1,floor(((var(8)+var(59)+var(11)+var(171)+var(457))/5)+1/3)+(2*var(7)-2)-max(0,(2*var(206)\n-var(211)))-var(237)/2)*max(0,(var(457)-1)))'}

In [58]:
m['Model']['Variables'][variableDict['Pax5']]

{'Name': 'Pax5',
 'Id': 11,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'min(2,floor(avg(var(74),var(8)))-var(275)/2 +2*var(437))-var(464)'}

In [63]:
m['Model']['Variables'][variableDict['Foxo1']]

{'Name': 'Foxo1',
 'Id': 8,
 'RangeFrom': 0,
 'RangeTo': 2,
 'Formula': 'floor(avg(var(74),(2*var(7)-2),var(17)))-var(307)'}

In [64]:
ebf1Var = ctx.variables.index(74)
pax5Var = ctx.variables.index(11)
foxo1Var = ctx.variables.index(8)

In [65]:
finding, trace = exists_order_before_online(qM, ctx, final_state_tuple, foxo1Var, 1, ebf1Var, 1)
print(finding, trace)

True [(0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 2, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0), (1, 2, 1,

In [66]:
report_trace(trace, ctx, m)

Trace length: 12 states, 11 transitions
Start: {'Ilr7': 0, 'Il7': 0, 'E2a': 0, 'Foxo1': 0, 'Pax5': 0, 'Ikzf1': 0, 'CD19': 0, 'BCR': 0, 'RAG': 0, 'RUNX1': 2, 'Vpreb1': 0, 'EBF1': 0, 'Proliferation': 1, 'Apoptosis': 1, 'Cycle-arrest': 1, 'CCND3': 0, 'Bcl2': 0, 'TSLP': 0, 'CLP': 1, 'PreproB': 0, 'ProB': 0, 'PreB': 0, 'Naive B cell': 0, 'JAK-STAT': 0, 'Syk': 0, 'Bcl6': 0, 'CDKN1A': 0, 'CDKN1B': 0, 'Bok': 0, 'preBCR': 0, 'Bach2': 0, 'CXCL12': 2, 'CXCR4': 0, 'Id3': 0, 'Ikzf3': 0, 'Igll1': 0, 'CRLF2': 0, 'Myc': 0, 'PIP3': 0, 'CCND2': 1, 'BCR-ABL': 0, 'CDKN2A': 1, 'preBCR_Pax5': 0, 'AKT': 0, 'DNA-damage': 0, 'Bim': 0, 'p53': 0, 'MAPK': 0, 'ERK1_2': 0, 'Blnk': 0, 'CXCR4_CXCL12': 0, 'KO_preBCR': 0, 'KO_JAK_STAT': 0, 'IKZF1_ko': 0, 'BCL6_KO': 0, 'MYC_KO': 0, 'KO_CDKN1A': 0, 'KO_CDKN1B': 0, 'KO_CDKN2A': 0, 'Force-Pax5': 0, 'EBF1_KO': 0, 'PAX5_KO': 0, 'KO_PIP3': 0, 'KO_MAPK': 0, 'KO_BCRABL': 0}

Step 1: Il7  0 -> 1
Step 2: Il7  1 -> 2
Step 3: Ikzf1  0 -> 1
Step 4: E2a  0 -> 1
Step 5: Ilr7  0 -> 1
S

In [61]:
def exists_order_before_dfs(
    qn,
    ctx: BNContext,
    start: tuple,
    i: int, v: int,
    j: int, w: int
) -> tuple[bool, list[tuple] | None]:
    """
    DFS version. Finds a witnessing trace faster on large models at the cost
    of returning a longer trace than BFS. Eliminates the parent dict by
    carrying the path explicitly on the stack.
    """
    start_code     = ctx.encoder.encode(start)
    initial_i_seen = start[i] == v

    visited: set[tuple[int, bool]] = set()

    # Stack carries (code, i_seen, path_so_far)
    # path is a list of decoded tuples — only the active branch is in memory
    stack = [(start_code, initial_i_seen, [start])]

    while stack:
        code, i_seen, path = stack.pop()

        if (code, i_seen) in visited:
            continue
        visited.add((code, i_seen))

        state = ctx.encoder.decode(code)

        if state[j] == w and not i_seen:
            continue

        if i_seen and state[j] != w:
            return True, path

        for successor in ctx.async_step(qn, ctx.tupleToState(state)):
            successor_code = ctx.encoder.encode(successor)
            new_i_seen     = i_seen or (successor[i] == v)
            key            = (successor_code, new_i_seen)
            if key not in visited:
                stack.append((successor_code, new_i_seen, path + [successor]))

    return False, None

In [62]:
finding, trace = exists_order_before_dfs(qM, ctx, final_state_tuple, pax5Var, 1, ebf1Var, 1)
print(finding, trace)

KeyboardInterrupt: 